# Load Packages

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

os.makedirs("data/features", exist_ok=True)

# Load Data

In [2]:
orders   = pd.read_csv("data/raw/olist_orders_dataset.csv", parse_dates=[
               "order_purchase_timestamp",
               "order_delivered_customer_date"])
reviews  = pd.read_csv("data/raw/olist_order_reviews_dataset.csv",
               parse_dates=["review_creation_date"])
items    = pd.read_csv("data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("data/raw/olist_order_payments_dataset.csv")
customers= pd.read_csv("data/raw/olist_customers_dataset.csv")
products = pd.read_csv("data/raw/olist_products_dataset.csv")
cat_map  = pd.read_csv("data/raw/product_category_name_translation.csv")

products = products.merge(cat_map, on="product_category_name", how="left")

# One review per order (keep lowest score)
rev = (reviews.sort_values("review_score")
       .drop_duplicates("order_id", keep="first")
       [["order_id", "review_score", "review_creation_date"]])

pay = (payments.groupby("order_id")
       .agg(total_value=("payment_value", "sum"),
            payment_type=("payment_type", "first"))
       .reset_index())

df = (orders
      .merge(customers[["customer_id", "customer_unique_id", "customer_state"]],
             on="customer_id", how="left")
      .merge(rev,  on="order_id", how="left")
      .merge(pay,  on="order_id", how="left"))

df["is_return"] = (
    (df["order_status"] == "delivered") & (df["review_score"] <= 2)
).astype(int)

df["days_to_review"] = (
    df["review_creation_date"] - df["order_delivered_customer_date"]
).dt.days
df["days_to_review"] = df["days_to_review"].where(df["days_to_review"] >= 0).clip(upper=90)

# Delivered orders only for feature computation
delivered = df[df["order_status"] == "delivered"].copy()
print(f"  {len(delivered):,} delivered orders across {delivered['customer_unique_id'].nunique():,} customers")

  96,478 delivered orders across 93,358 customers


In [4]:
delivered.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,review_score,review_creation_date,total_value,payment_type,is_return,days_to_review
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,SP,4.0,2017-10-11,38.71,credit_card,0,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,BA,4.0,2018-08-08,141.46,boleto,0,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,GO,5.0,2018-08-18,179.12,credit_card,0,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,RN,5.0,2017-12-03,72.20,credit_card,0,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,SP,5.0,2018-02-17,28.62,credit_card,0,0.0


## FEATURE 1 — Return rate + order volume + value

In [5]:
rate_feats = (delivered
    .groupby("customer_unique_id")
    .agg(
        total_orders      = ("order_id",    "count"),
        total_returns     = ("is_return",   "sum"),
        avg_order_value   = ("total_value", "mean"),
        max_order_value   = ("total_value", "max"),
        payment_type_mode = ("payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else "unknown"),
    )
    .reset_index())

rate_feats["return_rate"] = rate_feats["total_returns"] / rate_feats["total_orders"]

# Only customers with >= 2 orders (single-order customers are noise)
rate_feats = rate_feats[rate_feats["total_orders"] >= 2].copy()
print(f"  {len(rate_feats):,} customers with >= 2 orders")

  2,801 customers with >= 2 orders


## FEATURE 2 — Timing (days between delivery and review)

In [6]:
timing_base = delivered[
    delivered["days_to_review"].notna() &
    delivered["days_to_review"].gt(0)
].copy()

timing_feats = (timing_base
    .groupby("customer_unique_id")["days_to_review"]
    .agg(
        avg_days_to_review = "mean",
        min_days_to_review = "min",
        std_days_to_review = "std",
    )
    .reset_index())

# Flag: ever complained within 3 days of delivery
fast_complainers = (timing_base[timing_base["days_to_review"] <= 3]
    ["customer_unique_id"].unique())
timing_feats["fast_complainer"] = timing_feats["customer_unique_id"].isin(fast_complainers).astype(int)
timing_feats["std_days_to_review"] = timing_feats["std_days_to_review"].fillna(0)

## FEATURE 3 — Category return affinity (top 8 returned categories as flags)

In [7]:
orders_items = (delivered
    .merge(items[["order_id", "product_id"]], on="order_id", how="left")
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left"))

# Top 8 most-returned categories
top_cats = (orders_items[orders_items["is_return"] == 1]
    ["product_category_name"]
    .value_counts()
    .head(8)
    .index.tolist())

# For each customer: did they ever return in this category?
cat_flags = []
for cat in top_cats:
    mask = (orders_items["product_category_name"] == cat) & (orders_items["is_return"] == 1)
    flag = (orders_items[mask]
        .groupby("customer_unique_id")["is_return"]
        .max()
        .rename(f"returned_{cat.replace(' ', '_')}"))
    cat_flags.append(flag)

cat_feats = pd.concat(cat_flags, axis=1).fillna(0).astype(int).reset_index()


## ASSEMBLE

In [8]:
features = (rate_feats
    .merge(timing_feats, on="customer_unique_id", how="left")
    .merge(cat_feats,    on="customer_unique_id", how="left"))

# Payment type one-hot
pay_dummies = pd.get_dummies(features["payment_type_mode"], prefix="pay").astype(int)
features = pd.concat([features.drop(columns=["payment_type_mode"]), pay_dummies], axis=1)

features = features.fillna(0)

features.to_parquet("data/features/customer_features.parquet", index=False)
print(f"\nSaved: data/features/customer_features.parquet")
print(f"Shape: {features.shape}")
print(f"\nColumns:\n{features.columns.tolist()}")
print(f"\nReturn rate stats:\n{features['return_rate'].describe().round(3)}")


Saved: data/features/customer_features.parquet
Shape: (2801, 22)

Columns:
['customer_unique_id', 'total_orders', 'total_returns', 'avg_order_value', 'max_order_value', 'return_rate', 'avg_days_to_review', 'min_days_to_review', 'std_days_to_review', 'fast_complainer', 'returned_cama_mesa_banho', 'returned_moveis_decoracao', 'returned_informatica_acessorios', 'returned_beleza_saude', 'returned_esporte_lazer', 'returned_utilidades_domesticas', 'returned_relogios_presentes', 'returned_telefonia', 'pay_boleto', 'pay_credit_card', 'pay_debit_card', 'pay_voucher']

Return rate stats:
count    2801.000
mean        0.125
std         0.277
min         0.000
25%         0.000
50%         0.000
75%         0.000
max         1.000
Name: return_rate, dtype: float64
